# Stage 7 — Reporting (McKinsey Standard)
## Google Play Store Apps · Tier A

This notebook assembles `reports/final_report.html`. It **hardcodes no statistic** — every number is
read from `reports/_key_figures.json` and the CSVs written by Stages 5a and 5b, so the report cannot
drift from the analysis.

| Step | What happens |
|---|---|
| 7.1 | Finalize SCR → lock the Governing Thought → 4 MECE Key Lines |
| 7.2 | Render 12 exhibits to `reports/figures/` under the DESIGN.md system |
| 7.3 | Assemble a self-contained HTML: inline CSS, base64 images, no external requests |
| 7.4 | Run the Stage 7 + Tier A gate checklists against the artifact |

In [1]:
from __future__ import annotations

import base64, io, json, re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
FIGS = REPORTS / "figures"; FIGS.mkdir(parents=True, exist_ok=True)

K = json.loads((REPORTS / "_key_figures.json").read_text(encoding="utf-8"))
apps  = pd.read_parquet(PROC / "apps_app_grain.parquet")
bycat = pd.read_parquet(PROC / "apps_by_category.parquet")
clean = pd.read_parquet(PROC / "apps_clean.parquet")
reach_models  = pd.read_csv(REPORTS / "_reach_models.csv")
rating_models = pd.read_csv(REPORTS / "_rating_models.csv")
paid_tests    = pd.read_csv(REPORTS / "_paid_share_tests.csv")
cm_df         = pd.read_csv(REPORTS / "_confusion_matrix.csv", index_col=0)
audit         = pd.read_csv(REPORTS / "_fairness_audit.csv", index_col=0)
ctrl          = pd.read_csv(REPORTS / "_censoring_evidence.csv", index_col=0)
spec          = pd.read_csv(REPORTS / "_target_specification.csv")

MACRO = ["Niche (<1k)", "Emerging (1k-100k)", "Established (100k-10M)", "Blockbuster (10M+)"]
INSTALL_LADDER = ["0", "0+", "1+", "5+", "10+", "50+", "100+", "500+", "1,000+", "5,000+",
                  "10,000+", "50,000+", "100,000+", "500,000+", "1,000,000+", "5,000,000+",
                  "10,000,000+", "50,000,000+", "100,000,000+", "500,000,000+", "1,000,000,000+"]
print(f"{len(K)} key figures loaded; {len(apps):,} apps")
print({k: K[k] for k in ["rating_missing_pct", "hhi", "reach_macro_f1", "blockbuster_recall",
                          "rating_best_r2", "rating_verdict"]})

73 key figures loaded; 9,659 apps
{'rating_missing_pct': 15.1, 'hhi': 0.0727, 'reach_macro_f1': 0.503, 'blockbuster_recall': 0.261, 'rating_best_r2': 0.074, 'rating_verdict': 'REJECT'}


## 7.2 Design system — DESIGN.md, consumed by code

Palette tokens and rcParams are declared once here and used by every exhibit. **Colors are never
re-picked per chart** (DESIGN.md §6).

### Semantic mapping for this project

| Role | Token |
|---|---|
| the honest / corrected series | **Blue** `#2251FF` |
| the censored, leaked or survivor series | **Grey** `#9FADB8` (context — the thing being retired) |
| the validated alternative (reach tiers, structure) | **Teal** `#00857C` |
| baselines, pre-registered bars, thresholds | **Amber** `#C1841C` |
| secondary cuts | **Slate** `#7F93A6` |

### Validator obligations, carried through

The categorical set was re-run through the `dataviz` validator. Amber↔Teal sits at **tritan ΔE 6.0**,
and Grey is below 3:1 contrast on white. Both are legal **only with secondary encoding**, so every
multi-series exhibit carries a **legend *and* direct labels**, and Grey never identifies a series by
colour alone. No dual-axis charts anywhere.

In [2]:
NAVY, BLUE, TEAL   = "#051C2C", "#2251FF", "#00857C"
AMBER, SLATE, GREY = "#C1841C", "#7F93A6", "#9FADB8"
GRIDLINE = "#E9ECEF"

mpl.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
    "font.size": 9.5, "text.color": NAVY, "axes.labelcolor": NAVY,
    "xtick.color": SLATE, "ytick.color": SLATE, "axes.edgecolor": GRIDLINE, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlelocation": "left",
    "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.titlepad": 10,
    "axes.grid": True, "grid.color": GRIDLINE, "grid.linewidth": 0.7, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 8.5, "xtick.major.size": 0, "ytick.major.size": 0,
    "figure.dpi": 120, "savefig.dpi": 170, "savefig.bbox": "tight",
})
PCT = FuncFormatter(lambda v, _: f"{v:.0f}%")
EXHIBITS = {}          # id -> dict(title, source, so_what, implication, path)

def title_block(ax, title, source):
    ax.set_title(title, loc="left", color=NAVY, fontsize=11.5, fontweight="bold", pad=20)
    ax.text(0, 1.015, source, transform=ax.transAxes, ha="left", va="bottom",
            fontsize=8.2, color=SLATE)

def register(eid, fig, title, source, so_what, implication):
    p = FIGS / f"{eid}.png"
    fig.savefig(p, dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    EXHIBITS[eid] = {"title": title, "source": source, "so_what": so_what,
                     "implication": implication, "path": p}
    print(f"  {eid}  {title[:76]}")

print("design system applied; rendering exhibits...")

design system applied; rendering exhibits...


### Part I exhibits — the diagnostic (Key Lines 1–2)

In [3]:
# ================= EX 1 - HERO: the censoring staircase =========================================
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.7))
buckets = pd.cut(apps["Reviews_int"], [-1, 0, 10, 100, 1_000, 10_000, np.inf],
                 labels=["0", "1-10", "11-100", "101-1k", "1k-10k", ">10k"])
bs = apps.groupby(buckets, observed=True)["Rating_Observed"].agg(n="size", obs="sum")
bs["miss"] = (1 - bs["obs"] / bs["n"]) * 100
ax = axes[0]
ax.bar(range(len(bs)), bs["miss"], color=BLUE, width=0.62)
for i, v in enumerate(bs["miss"]):
    ax.text(i, v + 3, f"{v:.1f}%", ha="center", fontsize=8.5, color=NAVY, fontweight="bold")
ax.set_xticks(range(len(bs)))
ax.set_xticklabels([f"{i}\nn={n:,}" for i, n in zip(bs.index, bs["n"])], fontsize=7.8)
ax.set_ylim(0, 118); ax.yaxis.set_major_formatter(PCT); ax.set_xlabel("reviews", fontsize=9)
title_block(ax, "Zero reviews means no rating - always",
            "% of apps missing Rating, by review count")

ls = apps.groupby("Installs", observed=True)["Rating_Observed"].agg(n="size", obs="sum")
ls["miss"] = (1 - ls["obs"] / ls["n"]) * 100
ls = ls.reindex([b for b in INSTALL_LADDER if b in ls.index])
ax = axes[1]
ax.plot(range(len(ls)), ls["miss"], color=BLUE, lw=2.2, marker="o", ms=4.5,
        markerfacecolor="white", markeredgewidth=1.6, markeredgecolor=BLUE)
ax.fill_between(range(len(ls)), ls["miss"], color=BLUE, alpha=0.08)
ax.set_xticks(range(len(ls)))
ax.set_xticklabels([b.replace(",000,000", "M").replace(",000", "k").replace("+", "")
                    for b in ls.index], rotation=90, fontsize=6.8)
ax.set_ylim(-3, 108); ax.yaxis.set_major_formatter(PCT)
ax.set_xlabel("install bracket (all 21)", fontsize=9)
title_block(ax, "...and the rate decays monotonically to zero with scale",
            f"Cochran-Armitage trend z = {K['cochran_armitage_z']}")
plt.tight_layout()
register("ex01", fig,
         f"Google withholds the star rating below a review threshold - so the {K['rating_missing_pct']}% "
         "of apps with no rating are the catalog's long tail, not random gaps",
         f"Google Play Store catalog, vintage Aug 2018 | n = {K['n_apps_app_grain']:,} apps "
         "(one row per app) | Cochran-Armitage trend test across all 21 install brackets",
         f"The {K['rating_missing_n']:,} apps without a rating are not missing data to be dropped - "
         f"they are the smallest apps in the store, and they are absent by design. Every "
         f"'average rating by category' figure published from this dataset silently conditions on "
         f"apps that survived long enough to be rated.",
         "Rating statistics must carry their denominator, and Rating cannot be imputed - imputation "
         "would fabricate exactly the tail that is missing.")

# ================= EX 2 - the sample shift ======================================================
obs = apps["Rating_Observed"]
mix = pd.crosstab(apps["Installs_macro"], obs, normalize="columns").mul(100).reindex(MACRO)
fig, ax = plt.subplots(figsize=(8.6, 3.5))
y = np.arange(len(MACRO)); h = 0.36
ax.barh(y + h/2, mix[True], height=h, color=BLUE,
        label=f"Rating observed (n={int(obs.sum()):,})")
ax.barh(y - h/2, mix[False], height=h, color=GREY,
        label=f"Rating missing (n={int((~obs).sum()):,})")
for i, (a, b) in enumerate(zip(mix[True], mix[False])):
    ax.text(a + 1.2, i + h/2, f"{a:.1f}%", va="center", fontsize=8.4, color=NAVY, fontweight="bold")
    ax.text(b + 1.2, i - h/2, f"{b:.1f}%", va="center", fontsize=8.4, color=NAVY)
ax.set_yticks(y); ax.set_yticklabels(MACRO, fontsize=9); ax.invert_yaxis()
ax.set_xlim(0, 106); ax.xaxis.set_major_formatter(PCT); ax.legend(loc="lower right")
title_block(ax, "The unrated apps are almost entirely the niche tier",
            "Install-tier mix within each group | legend + direct labels")
plt.tight_layout()
register("ex02", fig,
         f"Dropping unrated apps deletes {K['unrated_pct_niche']}% of the niche tier and multiplies "
         f"median reviews from {K['median_reviews_unrated']:,.0f} to {K['median_reviews_rated']:,.0f}",
         f"Grain: app ({K['n_apps_app_grain']:,}) | Cliff's delta on installs = "
         f"{K['cliffs_delta_installs']} (very large)",
         f"The rated subsample has a median of {K['median_installs_rated']:,.0f} installs; the "
         f"unrated one has {K['median_installs_unrated']:,.0f}. These are two different populations, "
         "not one population with holes in it.",
         "Any category-quality comparison built on rated apps only describes incumbents. It cannot "
         "tell a studio what a NEW listing should expect.")

  ex01  Google withholds the star rating below a review threshold - so the 15.1% of 


  ex02  Dropping unrated apps deletes 91.8% of the niche tier and multiplies median 


In [4]:
# ================= EX 3 - THE CONTROL ============================================================
fig, ax = plt.subplots(figsize=(9.0, 3.6))
x = np.arange(len(MACRO)); w = 0.26
ax.bar(x - w, ctrl["rating_missing"], w, color=BLUE,  label="Rating missing")
ax.bar(x,     ctrl["size_varies"],    w, color=TEAL,  label="Size = 'Varies with device'")
ax.bar(x + w, ctrl["android_varies"], w, color=AMBER, label="Android Ver = 'Varies'")
for xi in range(len(MACRO)):
    for off, col in [(-w, "rating_missing"), (0, "size_varies"), (w, "android_varies")]:
        v = ctrl[col].iloc[xi]
        ax.text(xi + off, v + 1.6, f"{v:.0f}", ha="center", fontsize=7.8, color=NAVY,
                fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([m.split(" (")[0] for m in MACRO], fontsize=9)
ax.yaxis.set_major_formatter(PCT); ax.set_ylim(0, 100); ax.legend(loc="upper center", ncol=3)
title_block(ax, "Rating thins out as apps get smaller; the other optional fields do the opposite",
            "% of apps in each install tier | legend + direct labels")
plt.tight_layout()
register("ex03", fig,
         "The censoring is specific to Rating - it is not a general 'small apps have worse metadata' "
         "effect, because the other optional fields grade the opposite way",
         f"Spearman vs install tier: Rating missing {K['control_rho_rating_missing']:+.2f}, "
         f"Size 'Varies' {K['control_rho_size_varies']:+.2f}, "
         f"Android 'Varies' {K['control_rho_android_varies']:+.2f}",
         "The obvious challenge to Key Line 1 is that small apps simply have thinner metadata. If "
         "that were true, every optional field would thin out together. They do not: 'Varies with "
         "device' RISES with scale while Rating missingness FALLS.",
         "The finding survives its most likely objection, so the rating denominator is a real "
         "correction rather than an artifact of data quality.")

# ================= EX 4 - concentration ==========================================================
share = bycat["Category"].value_counts()
share_pct = share / share.sum() * 100
fig, ax = plt.subplots(figsize=(11.4, 3.5))
cols = [BLUE if i < 5 else GREY for i in range(len(share))]
ax.bar(range(len(share)), share.values, color=cols, width=0.72)
for i in range(5):
    ax.text(i, share.values[i] + 34, f"{share_pct.values[i]:.1f}%", ha="center", fontsize=8.4,
            color=NAVY, fontweight="bold")
ax.axhline(share.mean(), color=AMBER, lw=1.3, ls="--")
ax.text(len(share) - 0.5, share.mean() + 36, f"mean {share.mean():.0f}", ha="right", fontsize=8,
        color=AMBER, fontweight="bold")
ax.set_xticks(range(len(share)))
ax.set_xticklabels([c.replace("_AND_", "&").replace("_", " ").title() for c in share.index],
                   rotation=90, fontsize=6.8)
ax.set_ylabel("apps"); ax.set_ylim(0, share.max() * 1.18)
title_block(ax, f"Top 5 of {K['n_categories']} categories hold {K['top5_share_pct']}% of listings",
            f"HHI = {K['hhi']} | blue = top 5 | amber = mean")
plt.tight_layout()
register("ex04", fig,
         f"Crowding is not the entry constraint - at HHI {K['hhi']} the catalog is unconcentrated by "
         "any competition-policy standard",
         f"Grain: (App, Category) {K['n_apps_by_category']:,} - an app listed in two categories "
         "counts in both | Herfindahl-Hirschman Index over category shares",
         f"{K['largest_category']} leads at {K['largest_category_share_pct']}%, and HHI "
         f"{K['hhi']} sits far below the 0.15 'unconcentrated' threshold. The intuition that the "
         "store is dominated by a few categories is not supported.",
         "Category choice should not be driven by avoiding crowded categories. The differentiating "
         "factor is monetization structure (EX 5), not headcount.")

  ex03  The censoring is specific to Rating - it is not a general 'small apps have w


  ex04  Crowding is not the entry constraint - at HHI 0.0727 the catalog is unconcen


In [5]:
# ================= EX 5 - paid share with CIs =====================================================
sel = paid_tests.head(14).iloc[::-1]
fig, ax = plt.subplots(figsize=(9.0, 4.2))
ypos = np.arange(len(sel))
cols = [BLUE if s else GREY for s in sel["significant"]]
ax.barh(ypos, sel["paid_pct"], color=cols, height=0.6)
ax.hlines(ypos, sel["ci_lo"], sel["ci_hi"], color=NAVY, lw=1.4)
ax.axvline(K["paid_share_overall_pct"], color=AMBER, lw=1.4, ls="--")
ax.text(K["paid_share_overall_pct"] + 0.5, len(sel) - 0.5,
        f"catalog {K['paid_share_overall_pct']}%", fontsize=8.4, color=AMBER, fontweight="bold")
for i in range(len(sel)):
    ax.text(sel["ci_hi"].iloc[i] + 0.6, i, f"{sel['paid_pct'].iloc[i]:.1f}%", va="center",
            fontsize=8.4, color=NAVY, fontweight="bold")
ax.set_yticks(ypos)
ax.set_yticklabels([c.replace("_AND_", "&").replace("_", " ").title() for c in sel["Category"]],
                   fontsize=8.4)
ax.xaxis.set_major_formatter(PCT); ax.set_xlim(0, 32)
title_block(ax, "Paid viability separates by category, with CIs clear of the catalog average",
            "% priced > $0, Wilson 95% CI | blue = differs after Holm correction | n >= 40")
plt.tight_layout()
register("ex05", fig,
         f"Monetization is a category property, not an app decision - {K['paid_share_max_cat']} "
         f"sustains {K['paid_share_max_pct']}% paid listings while several categories sustain none",
         f"Grain: (App, Category) {K['n_apps_by_category']:,} | Fisher exact vs rest of catalog, "
         "Holm-corrected within the 33-category family | Wilson 95% CIs",
         f"Against a {K['paid_share_overall_pct']}% catalog average, "
         + " and ".join(f"{r.Category} ({r.paid_pct:.1f}%)"
                        for r in paid_tests.head(2).itertuples())
         + " run above 20%, while "
         + ", ".join(paid_tests.loc[paid_tests['paid_pct'] == 0, 'Category'].tolist())
         + " sustain none. The confidence intervals do not overlap the average, so this is "
           "structure, not sampling noise.",
         "Choose the monetization model WITH the category, not after it. A paid launch into a "
         "0%-paid category bets against every incumbent's revealed pricing.")

# ================= EX 6 - price distribution ======================================================
paid = apps[apps["Price_numeric"] > 0]
non_nov = paid[~paid["Is_Novelty_Price"]]
fig, ax = plt.subplots(figsize=(8.8, 3.4))
ax.hist(non_nov["Price_numeric"].clip(upper=20), bins=np.arange(0, 20.5, 0.5), color=BLUE,
        edgecolor="white", lw=0.5)
med, mean = K["paid_price_median"], K["paid_price_mean"]
ax.axvline(med, color=NAVY, lw=1.8)
ax.text(med + 0.3, ax.get_ylim()[1] * 0.90, f"median ${med:.2f}", fontsize=9, color=NAVY,
        fontweight="bold")
ax.axvline(mean, color=AMBER, lw=1.8, ls="--")
ax.text(mean + 0.3, ax.get_ylim()[1] * 0.68,
        f"mean ${mean:.2f}\ndragged by {K['n_novelty_apps']} novelty\nlistings up to $400",
        fontsize=8.4, color=AMBER, fontweight="bold")
ax.set_xlabel("list price (USD, clipped at $20)"); ax.set_ylabel("paid apps")
title_block(ax, "Paid pricing clusters below $5", f"{len(paid):,} paid apps | grain: app")
plt.tight_layout()
register("ex06", fig,
         f"Half of all paid apps sell below ${med:.2f}, and the ${mean:.2f} mean is an artifact of "
         f"{K['n_novelty_apps']} novelty listings priced up to $400",
         f"Grain: app ({K['n_apps_app_grain']:,}) | novelty listings FLAGGED, not deleted - their "
         "effect is shown rather than hidden by a silent filter",
         f"The mean-to-median gap is {mean/med:.1f}x, created entirely by the 'I Am Rich' family of "
         "joke apps. They are real listings and are retained; the statistic is changed instead.",
         f"Price a new paid app in the $0.99-$4.99 band where 75% of paid listings sit. Never quote "
         "a mean price from this catalog.")

  ex05  Monetization is a category property, not an app decision - PERSONALIZATION s


  ex06  Half of all paid apps sell below $2.99, and the $14.06 mean is an artifact o


In [6]:
# ================= EX 7 - content rating ==========================================================
ct = pd.crosstab(bycat["Category"], bycat["Content Rating"])
ev = (ct.T / ct.sum(axis=1)).T["Everyone"] * 100
ev = ev[ct.sum(axis=1) >= 40].sort_values()
fig, ax = plt.subplots(figsize=(11.4, 3.4))
cols = [BLUE if v < 60 else GREY for v in ev]
ax.bar(range(len(ev)), ev.values, color=cols, width=0.72)
ax.axhline(K["everyone_share_pct"], color=AMBER, lw=1.4, ls="--")
ax.text(len(ev) - 0.4, K["everyone_share_pct"] + 2, f"catalog {K['everyone_share_pct']}%",
        ha="right", fontsize=8.4, color=AMBER, fontweight="bold")
for i in range(5):
    ax.text(i, ev.values[i] + 2, f"{ev.values[i]:.0f}%", ha="center", fontsize=8.4, color=NAVY,
            fontweight="bold")
ax.set_xticks(range(len(ev)))
ax.set_xticklabels([c.replace("_AND_", "&").replace("_", " ").title() for c in ev.index],
                   rotation=90, fontsize=6.8)
ax.yaxis.set_major_formatter(PCT); ax.set_ylim(0, 108)
title_block(ax, 'Dating rates 9% "Everyone" against a catalog norm above 80%',
            f"Cramer's V = {K['cramers_v_content_category']} | n >= 40")
plt.tight_layout()
register("ex07", fig,
         "Content gating is a structural reach ceiling that varies sharply by category",
         f"Grain: (App, Category) {K['n_apps_by_category']:,} | chi-square with bias-corrected "
         f"Cramer's V = {K['cramers_v_content_category']}",
         f'{K["everyone_share_pct"]}% of the catalog is rated "Everyone", but DATING sits at 9% and '
         "SOCIAL and ENTERTAINMENT below 40%. Audience access is set at category level.",
         "A category whose norm is Teen-or-above carries a reach ceiling no listing optimisation "
         "removes. Factor it into the addressable-market estimate before committing.")

# ================= EX 8 - size & recency ==========================================================
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.4))
sz = apps.dropna(subset=["Size_MB"])
m1 = sz.groupby("Installs_macro", observed=True)["Size_MB"].median().reindex(MACRO)
ax = axes[0]
ax.bar(range(4), m1.values, color=BLUE, width=0.6)
for i, v in enumerate(m1.values):
    ax.text(i, v + 0.7, f"{v:.0f} MB", ha="center", fontsize=9, color=NAVY, fontweight="bold")
ax.set_xticks(range(4)); ax.set_xticklabels([m.split(" (")[0] for m in MACRO], fontsize=8.5)
ax.set_ylabel("median size (MB)"); ax.set_ylim(0, m1.max() * 1.25)
title_block(ax, f"Reach rises with app size (rho {K['spearman_size_installs']:+.2f})",
            f"n = {len(sz):,} with a parsed size")
m2 = apps.groupby("Installs_macro", observed=True)["Days_Since_Update"].median().reindex(MACRO)
ax = axes[1]
ax.bar(range(4), m2.values, color=TEAL, width=0.6)
for i, v in enumerate(m2.values):
    ax.text(i, v + 8, f"{v:.0f} d", ha="center", fontsize=9, color=NAVY, fontweight="bold")
ax.set_xticks(range(4)); ax.set_xticklabels([m.split(" (")[0] for m in MACRO], fontsize=8.5)
ax.set_ylabel("median days since update"); ax.set_ylim(0, m2.max() * 1.25)
title_block(ax, f"and falls with staleness (rho {K['spearman_stale_installs']:+.2f})",
            f"n = {K['n_apps_app_grain']:,}")
plt.tight_layout()
register("ex08", fig,
         "Reach travels with maintenance and payload size - the two strongest listing associations "
         "in the catalog, and both are associations only",
         f"Grain: app ({K['n_apps_app_grain']:,}) | Spearman rank correlation vs install tier",
         "Blockbusters are a median 40 MB and updated within a fortnight; niche apps are ~8 MB and "
         "six months stale. No other listing attribute comes close to these two.",
         "ASSOCIATION, NOT CAUSE. Popular apps can afford maintenance, so causality plausibly runs "
         "backwards. 'Update more often to grow' is not supported by this data and is not "
         "recommended.")

  ex07  Content gating is a structural reach ceiling that varies sharply by category


  ex08  Reach travels with maintenance and payload size - the two strongest listing 


In [7]:
# ================= EX 9 - rating compression + falsification =======================================
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.5), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
r = apps["Rating_num"].dropna()
counts, _, _ = ax.hist(r, bins=np.arange(1, 5.11, 0.1), color=BLUE, edgecolor="white", lw=0.4)
ax.set_ylim(0, counts.max() * 1.3)
ax.axvspan(4.0, 4.7, color=AMBER, alpha=0.10, zorder=0)
ax.text(4.35, counts.max() * 1.17, f"{((r >= 4.0) & (r <= 4.7)).mean()*100:.0f}% of rated apps",
        ha="center", fontsize=8.4, color=AMBER, fontweight="bold")
ax.axvline(r.median(), color=NAVY, lw=1.6)
ax.set_xlabel("star rating"); ax.set_ylabel("apps"); ax.set_xlim(1, 5.1)
title_block(ax, f"Ratings compress into half a star (SD {K['rating_sd']})",
            f"n = {K['rating_subsample_n']:,} rated apps; {K['rating_missing_n']:,} unrated EXCLUDED")

ax = axes[1]
rd = rating_models.sort_values("r2")
cols = [GREY if m == "Dummy (mean)" else BLUE for m in rd["model"]]
ax.barh(range(len(rd)), rd["r2"], color=cols, height=0.6)
for i, v in enumerate(rd["r2"]):
    ax.text(v + 0.004, i, f"{v:+.3f}", va="center", fontsize=8.2, color=NAVY, fontweight="bold")
ax.axvline(K["rating_bar_preregistered"], color=AMBER, lw=2, ls="--")
ax.text(K["rating_bar_preregistered"] - 0.004, 0.2,
        f"pre-registered bar {K['rating_bar_preregistered']:.2f}", fontsize=8.2, color=AMBER,
        fontweight="bold", va="bottom", ha="right", rotation=90)
ax.set_yticks(range(len(rd))); ax.set_yticklabels(rd["model"], fontsize=7.8)
ax.set_xlim(-0.015, K["rating_bar_preregistered"] * 1.3); ax.set_xlabel("cross-validated R-squared")
title_block(ax, f"{K['rating_n_models_tested']} tuned models miss a deliberately low bar",
            f"verdict: {K['rating_verdict']}")
plt.tight_layout()
register("ex09", fig,
         f"App rating is not predictable from a listing - {K['rating_n_models_tested']} tuned models "
         f"peak at R-squared {K['rating_best_r2']} against a bar of "
         f"{K['rating_bar_preregistered']} fixed before any model ran",
         f"Rated subsample n = {K['rating_subsample_n']:,} | 5-fold CV | pre-launch features only "
         f"| category explains {K['eta2_rating_category']*100:.1f}% of rating variance",
         f"The target is both compressed (SD {K['rating_sd']} stars, 70% inside a 0.7-star band) and "
         "censored (EX 1). Even the winning model explains under 8% of what little variance exists.",
         "Remove rating forecasts from the greenlight scorecard. Rating is an outcome to monitor "
         "after launch, not a variable to plan with.")

# ================= EX 10 - the leakage exhibit =====================================================
fig, ax = plt.subplots(figsize=(7.4, 3.4))
lab = ["Dummy\n(mean)", "Ridge", "HistGB\nhonest", "HistGB\n+ Reviews"]
val = [0.0, K["r2_installs_ridge"], K["r2_installs_honest"], K["r2_installs_with_reviews"]]
cols = [SLATE, TEAL, BLUE, GREY]
ax.bar(range(4), val, color=cols, width=0.6)
for i, v in enumerate(val):
    ax.text(i, v + 0.028, f"{v:.3f}", ha="center", fontsize=9.5, color=NAVY, fontweight="bold")
ax.annotate("", xy=(3, K["r2_installs_with_reviews"] - 0.02), xytext=(3, K["r2_installs_honest"]),
            arrowprops=dict(arrowstyle="<->", color=AMBER, lw=1.6))
ax.text(3.32, (K["r2_installs_with_reviews"] + K["r2_installs_honest"]) / 2,
        "the leak", fontsize=8.6, color=AMBER, fontweight="bold", va="center")
ax.set_xticks(range(4)); ax.set_xticklabels(lab, fontsize=8.5)
ax.set_ylabel("cross-validated R-squared"); ax.set_ylim(0, 1.06)
title_block(ax, "Feeding Reviews back in more than doubles R-squared",
            "Target: log10(install floor) | 5-fold CV | grain: app")
plt.tight_layout()
register("ex10", fig,
         f"Reviews reproduces {K['reviews_installs_r2']*100:.0f}% of install variance on its own - a "
         "model that keeps it is scoring a tautology, not a forecast",
         f"Spearman(log Reviews, install tier) = {K['reviews_installs_spearman']} | "
         f"grain: app ({K['n_apps_app_grain']:,})",
         f"With Reviews the model scores R-squared {K['r2_installs_with_reviews']}; without it, "
         f"{K['r2_installs_honest']}. Reviews only exists after launch, so it cannot inform a "
         "pre-launch decision at all.",
         "Published notebooks reporting ~0.9 on this dataset are measuring the outcome against "
         "itself. Every model in this report excludes Reviews and its derivatives.")

  ex09  App rating is not predictable from a listing - 8 tuned models peak at R-squa


  ex10  Reviews reproduces 91% of install variance on its own - a model that keeps i


In [8]:
# ================= EX 11 - formulation + bake-off ==================================================
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.6), gridspec_kw={"width_ratios": [1, 1.15]})
ax = axes[0]
labs = [f"{K['n_brackets_tested']}-class\nraw brackets", "4-class\ndecade tiers"]
f1s = [K["macro_f1_21class"], K["reach_macro_f1"]]
ax.bar([0, 1], f1s, color=[GREY, BLUE], width=0.5)
for i, v in enumerate(f1s):
    ax.text(i, v + 0.018, f"{v:.3f}", ha="center", fontsize=10, color=NAVY, fontweight="bold")
ax.axhline(K["reach_baseline_majority_f1"], color=AMBER, lw=1.5, ls="--")
ax.text(1.42, K["reach_baseline_majority_f1"] + 0.016, "majority baseline", ha="right",
        fontsize=8.2, color=AMBER, fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels(labs, fontsize=8.8)
ax.set_ylabel("macro-F1"); ax.set_ylim(0, 0.62)
title_block(ax, "The target specification decides whether reach is learnable",
            "Same features, same algorithm, 5-fold CV")

ax = axes[1]
rm = reach_models.sort_values("macro_f1")
cols = [GREY if m.startswith(("0a", "0b")) else (BLUE if m.startswith("3t") else TEAL)
        for m in rm["model"]]
ax.barh(range(len(rm)), rm["macro_f1"], color=cols, height=0.62)
for i, v in enumerate(rm["macro_f1"]):
    ax.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=8.2, color=NAVY, fontweight="bold")
ax.axvline(K["reach_baseline_majority_f1"], color=AMBER, lw=1.4, ls="--")
ax.set_yticks(range(len(rm))); ax.set_yticklabels(rm["model"], fontsize=7.6)
ax.set_xlim(0, rm["macro_f1"].max() * 1.2); ax.set_xlabel("macro-F1 (5-fold CV)")
title_block(ax, "Seven specifications, tuned - boosting wins narrowly",
            "Reviews EXCLUDED | grain: app")
plt.tight_layout()
register("ex11", fig,
         f"Pre-launch reach IS forecastable - macro-F1 {K['reach_macro_f1']} against a "
         f"{K['reach_baseline_majority_f1']} baseline ({K['reach_lift_vs_majority']}x) - but only at "
         "decade-tier resolution",
         f"Grain: app ({K['n_apps_app_grain']:,}) | stratified 5-fold CV | two baselines: majority "
         f"({K['reach_baseline_majority_f1']}) and stratified ({K['reach_baseline_stratified_f1']})",
         f"At the raw {K['n_brackets_tested']}-bracket resolution the same features reach macro-F1 "
         f"{K['macro_f1_21class']} - the model names the modal bracket and nothing else. Collapsed "
         "to four decade-tiers it becomes genuinely useful.",
         "Forecast reach as an order of magnitude, never as an install count. The install field is "
         "a bucket floor, so precision beyond a decade is not available in principle.")

# ================= EX 12 - confusion matrix ========================================================
cm = cm_df.values
recalls = cm.diagonal() / cm.sum(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.9), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=10,
                color="white" if cm[i, j] > cm.max() * 0.55 else NAVY,
                fontweight="bold" if i == j else "normal")
ax.set_xticks(range(4)); ax.set_xticklabels([m.split(" (")[0] for m in MACRO], rotation=22,
                                            ha="right", fontsize=8.4)
ax.set_yticks(range(4)); ax.set_yticklabels([m.split(" (")[0] for m in MACRO], fontsize=8.4)
ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.grid(False)
ax.add_patch(plt.Rectangle((-0.5, 2.5), 4, 1, fill=False, edgecolor=AMBER, lw=2.4))
title_block(ax, "Raw counts - the bottom row is the business limit", "5-fold CV predictions")
ax = axes[1]
ax.barh(range(4), recalls * 100, color=[TEAL, TEAL, TEAL, AMBER], height=0.6)
for i, v in enumerate(recalls * 100):
    ax.text(v + 1.4, i, f"{v:.0f}%", va="center", fontsize=9.5, color=NAVY, fontweight="bold")
ax.set_yticks(range(4)); ax.set_yticklabels([m.split(" (")[0] for m in MACRO], fontsize=8.4)
ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(PCT); ax.set_xlabel("recall")
title_block(ax, "Recall collapses on the tier that matters most", "amber = Blockbuster")
plt.tight_layout()
register("ex12", fig,
         f"The model finds only {K['blockbuster_recall']:.0%} of blockbusters - "
         f"{K['blockbuster_called_established']} of {K['blockbuster_n']} are called 'Established' - "
         "so it can band a portfolio but must never pick winners",
         f"Grain: app ({K['n_apps_app_grain']:,}) | macro-F1 {K['reach_macro_f1']} "
         f"(bootstrap 95% CI {K['reach_macro_f1_ci']})",
         f"Aggregate performance is real, but {K['blockbuster_missed']} of {K['blockbuster_n']} "
         "top-tier apps are missed, almost all mistaken for the tier below. The model cannot see "
         "what separates a hit from a solid performer.",
         "Use the model to BAND expected reach for budgeting. Do NOT use it to select which app to "
         "greenlight - it would reject three-quarters of the eventual hits.")

  ex11  Pre-launch reach IS forecastable - macro-F1 0.503 against a 0.132 baseline (


  ex12  The model finds only 26% of blockbusters - 294 of 437 are called 'Establishe


In [9]:
# ================= EX 13 - fairness audit ==========================================================
a = audit.sort_values("under_predicted_pct")
fig, ax = plt.subplots(figsize=(11.4, 3.4))
thresh = a["under_predicted_pct"].mean() + 8
ax.bar(range(len(a)), a["under_predicted_pct"],
       color=[AMBER if v > thresh else BLUE for v in a["under_predicted_pct"]], width=0.72)
ax.axhline(a["under_predicted_pct"].mean(), color=SLATE, lw=1.3, ls="--")
ax.text(len(a) - 0.4, a["under_predicted_pct"].mean() + 1.4,
        f"mean {a['under_predicted_pct'].mean():.1f}%", ha="right", fontsize=8.4, color=SLATE,
        fontweight="bold")
ax.set_xticks(range(len(a)))
ax.set_xticklabels([c.replace("_AND_", "&").replace("_", " ").title() for c in a.index],
                   rotation=90, fontsize=6.8)
ax.yaxis.set_major_formatter(PCT); ax.set_ylabel("% of apps banded too low")
title_block(ax, "Under-prediction is not evenly spread across categories",
            "% assigned a LOWER tier than actual | amber = >8pp above mean | n >= 40")
plt.tight_layout()
register("ex13", fig,
         "If the reach model informs budget, its per-category error must be monitored - some "
         "categories are systematically banded too low",
         f"Grain: app ({K['n_apps_app_grain']:,}) | per-category accuracy ranges "
         f"{K['audit_acc_min']}-{K['audit_acc_max']}",
         f"Under-prediction averages {K['audit_underpred_mean']}% but varies widely by category. A "
         "model used to allocate budget would give systematically conservative estimates to the "
         "categories at the high end.",
         "Monitor per-category error rates in production and manually review the high-under-"
         "prediction categories. Otherwise the model quietly entrenches existing category size.")

print(f"\n{len(EXHIBITS)} exhibits rendered to reports/figures/")

  ex13  If the reach model informs budget, its per-category error must be monitored 

13 exhibits rendered to reports/figures/


## 7.3 Assemble the HTML

Self-contained by construction: all CSS inline, all figures base64-embedded, **zero external
requests**. Theme is token-driven — `:root`, then `@media (prefers-color-scheme: dark)`, then
`:root[data-theme="light"|"dark"]` so a viewer's toggle wins in both directions.

In [10]:
def b64(p: Path) -> str:
    return base64.b64encode(p.read_bytes()).decode("ascii")

def exhibit_html(eid, number):
    e = EXHIBITS[eid]
    return f"""
<figure class="figcard">
  <div class="exlabel">Exhibit {number}</div>
  <h3 class="extitle">{e['title']}</h3>
  <img src="data:image/png;base64,{b64(e['path'])}" alt="{e['title']}">
  <figcaption class="source">{e['source']}</figcaption>
</figure>
<div class="sowhat">
  <p><span class="lbl">So what</span>{e['so_what']}</p>
  <p><span class="lbl">Implication</span>{e['implication']}</p>
</div>
"""

CSS = r"""
:root{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#3d5566; --muted:#7F93A6;
  --accent:#2251FF; --accent-soft:#eef2ff; --rule:#dfe5ea; --teal:#00857C; --gold:#8A5E12;
  --gold-soft:#fdf6e6; --figcard:#ffffff;
}
@media (prefers-color-scheme: dark){
  :root{
    --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#b6c4ce; --muted:#8fa3b3;
    --accent:#5a8dff; --accent-soft:#12314a; --rule:#1d3d52; --teal:#3aa79e; --gold:#E5B84A;
    --gold-soft:#2a2312; --figcard:#ffffff;
  }
}
:root[data-theme="light"]{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#3d5566; --muted:#7F93A6;
  --accent:#2251FF; --accent-soft:#eef2ff; --rule:#dfe5ea; --teal:#00857C; --gold:#8A5E12;
  --gold-soft:#fdf6e6; --figcard:#ffffff;
}
:root[data-theme="dark"]{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#b6c4ce; --muted:#8fa3b3;
  --accent:#5a8dff; --accent-soft:#12314a; --rule:#1d3d52; --teal:#3aa79e; --gold:#E5B84A;
  --gold-soft:#2a2312; --figcard:#ffffff;
}
*{box-sizing:border-box}
body{
  margin:0; padding:0 20px 100px; background:var(--ground); color:var(--ink);
  font-family:'Helvetica Neue',Arial,system-ui,'Segoe UI',sans-serif;
  font-size:16px; line-height:1.62; -webkit-font-smoothing:antialiased;
}
.wrap{max-width:760px; margin:0 auto}
header{padding:64px 0 14px}
.eyebrow{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:11px; letter-spacing:.14em;
  text-transform:uppercase; color:var(--muted); margin-bottom:14px;
}
h1{
  font-family:Georgia,'Times New Roman',serif; font-weight:400; font-size:39px; line-height:1.2;
  margin:0 0 12px; text-wrap:balance; letter-spacing:-.01em;
}
.standfirst{font-size:17px; color:var(--ink-2); margin:0 0 6px; max-width:64ch}
.byline{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:11px; color:var(--muted);
  letter-spacing:.05em; margin-top:20px; padding-top:14px; border-top:1px solid var(--rule);
}
.answer{
  background:var(--surface); border-left:4px solid var(--accent); border-radius:0 4px 4px 0;
  padding:32px 34px; margin:34px 0 40px; box-shadow:0 1px 3px rgba(5,28,44,.06);
}
.answer h2{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:11px; letter-spacing:.14em;
  text-transform:uppercase; color:var(--muted); margin:0 0 22px; font-weight:400;
}
.scr{margin:0 0 26px}
.scr div{display:flex; gap:14px; margin-bottom:13px; align-items:baseline}
.scr .k{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:10px; letter-spacing:.1em;
  text-transform:uppercase; color:var(--accent); min-width:92px; flex-shrink:0; font-weight:700;
}
.scr .v{margin:0; font-size:15px; color:var(--ink-2)}
.gt{
  font-family:Georgia,'Times New Roman',serif; font-size:22px; line-height:1.42;
  color:var(--ink); background:var(--accent-soft); border-radius:4px;
  padding:24px 26px; margin:0 0 28px; text-wrap:balance;
}
.keylines{counter-reset:kl; list-style:none; padding:0; margin:0 0 26px}
.keylines li{counter-increment:kl; position:relative; padding-left:40px; margin-bottom:17px; font-size:15px}
.keylines li::before{
  content:counter(kl); position:absolute; left:0; top:1px;
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:12px; font-weight:700;
  color:var(--accent); border:1.5px solid var(--accent); border-radius:50%;
  width:24px; height:24px; display:grid; place-items:center;
}
.rec{background:var(--gold-soft); border-left:3px solid var(--gold); border-radius:0 4px 4px 0; padding:22px 24px}
.rec .k{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:10px; letter-spacing:.12em;
  text-transform:uppercase; color:var(--gold); font-weight:700; display:block; margin-bottom:10px;
}
.rec p{margin:0 0 9px; font-size:15px}
.rec p:last-child{margin-bottom:0}
.tiles{display:grid; grid-template-columns:repeat(4,1fr); gap:14px; margin:0 0 44px}
.tile{background:var(--surface); border-radius:4px; padding:20px 16px; border-top:2px solid var(--accent)}
.tile .n{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:27px; font-weight:700;
  color:var(--accent); font-variant-numeric:tabular-nums; line-height:1.1; display:block;
}
.tile .l{font-size:11.5px; color:var(--ink-2); margin-top:9px; display:block; line-height:1.4}
.part{margin:66px 0 30px; border-top:2px solid var(--accent); padding-top:14px}
.part .eyebrow{margin:0 0 6px}
.part h2{font-size:25px; margin:0; font-weight:600; letter-spacing:-.01em; text-wrap:balance}
h3.kl{font-size:19px; margin:44px 0 6px; font-weight:600; line-height:1.36; text-wrap:balance}
h3.kl .num{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:11px; color:var(--accent);
  letter-spacing:.1em; display:block; margin-bottom:7px; font-weight:700;
}
p{margin:0 0 15px}
.figcard{background:var(--figcard); border-radius:4px; padding:22px 22px 16px; margin:24px 0 0; border:1px solid var(--rule)}
.figcard img{width:100%; height:auto; display:block; margin:6px 0 0}
.exlabel{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:10px; letter-spacing:.12em;
  text-transform:uppercase; color:#7F93A6; margin-bottom:8px;
}
.extitle{font-size:15px; margin:0 0 4px; color:#051C2C; font-weight:600; line-height:1.4; text-wrap:balance}
.source{font-size:11px; color:#7F93A6; margin-top:12px; padding-top:10px; border-top:1px solid #E9ECEF; line-height:1.5}
.sowhat{border-left:3px solid var(--accent); background:var(--surface); padding:16px 20px; margin:0 0 12px; border-radius:0 4px 4px 0}
.sowhat p{margin:0 0 9px; font-size:14px; color:var(--ink-2)}
.sowhat p:last-child{margin:0}
.sowhat .lbl{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:10px; letter-spacing:.1em;
  text-transform:uppercase; color:var(--accent); font-weight:700; margin-right:9px;
}
.tablewrap{overflow-x:auto; margin:22px 0}
table{border-collapse:collapse; width:100%; font-size:13.5px; min-width:460px}
th{
  font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:10px; letter-spacing:.08em;
  text-transform:uppercase; color:var(--muted); text-align:left; padding:9px 12px;
  border-bottom:1.5px solid var(--rule); font-weight:400;
}
td{padding:9px 12px; border-bottom:1px solid var(--rule); font-variant-numeric:tabular-nums; color:var(--ink-2)}
td.num{text-align:right}
tr.best td{background:var(--accent-soft); color:var(--ink)}
.appendix{margin-top:70px; border-top:1px solid var(--rule); padding-top:34px}
.appendix h3{font-size:16px; margin:30px 0 9px; font-weight:600}
.appendix p,.appendix li{font-size:14px; color:var(--ink-2)}
.appendix ul{padding-left:20px; margin:0 0 15px}
.appendix li{margin-bottom:7px}
.limits li{margin-bottom:10px}
.check{list-style:none; padding:0}
.check li{padding-left:26px; position:relative; margin-bottom:6px; font-size:13.5px}
.check li::before{content:"\2713"; position:absolute; left:0; color:var(--teal); font-weight:700}
footer{margin-top:60px; padding-top:22px; border-top:1px solid var(--rule); font-size:12px; color:var(--muted)}
code{font-family:ui-monospace,'SF Mono',Consolas,monospace; font-size:.88em; background:var(--accent-soft); padding:1px 5px; border-radius:3px}
@media (max-width:680px){
  .tiles{grid-template-columns:repeat(2,1fr)}
  h1{font-size:31px} .gt{font-size:19px} body{padding:0 16px 70px}
  .scr div{flex-direction:column; gap:3px} .scr .k{min-width:0}
}
"""
print(f"CSS: {len(CSS):,} chars")

CSS: 7,328 chars


### 7.1 Storyline — SCR, Governing Thought, Key Lines

The Key Lines map one-to-one onto the issue-tree branches, which is what makes them MECE:
**A** = is the measure valid · **B** = what is structurally true · **C** = what associates with what ·
**D** = what can be predicted. Reading the four titles alone tells the whole argument (vertical logic).

In [11]:
gt = ("Enter on monetization structure rather than crowding, band expected reach with a model that "
      "must never be used to pick winners, and drop app rating from the greenlight scorecard "
      "entirely - it cannot be forecast, and the field itself is censored.")

scr = [
    ("Situation",
     "A studio choosing its next build needs to know which Play Store categories are crowded, which "
     "sustain paid distribution, and what reach a new listing can realistically expect. This "
     f"{K['n_apps_app_grain']:,}-app catalog snapshot is the most widely used public dataset for "
     "exactly that question."),
    ("Complication",
     f"Two of its headline fields do not mean what published analyses assume. Rating is missing for "
     f"{K['rating_missing_pct']}% of apps <em>because Google withholds it below a review "
     f"threshold</em> - those apps have a median of {K['median_reviews_unrated']:.0f} review and "
     f"{K['median_installs_unrated']:,.0f} installs - so every 'average rating by category' figure is "
     f"a survivor statistic. And Reviews, the feature most notebooks feed to install models, "
     f"reproduces {K['reviews_installs_r2']*100:.0f}% of install variance on its own."),
    ("Resolution",
     f"Corrected, the catalog still answers the entry question - but differently than expected. "
     f"Crowding is not the constraint (HHI {K['hhi']}); monetization structure is "
     f"({K['paid_share_max_cat']} {K['paid_share_max_pct']}% paid vs 0% in three categories). "
     f"Pre-launch reach is forecastable at macro-F1 {K['reach_macro_f1']} against a "
     f"{K['reach_baseline_majority_f1']} baseline - enough to band a portfolio, not to pick winners "
     f"({K['blockbuster_recall']:.0%} blockbuster recall). Rating is unforecastable "
     f"(R-squared {K['rating_best_r2']} against a pre-registered {K['rating_bar_preregistered']} bar)."),
]

key_lines = [
    f"<strong>The catalog's quality metric is structurally censored.</strong> "
    f"{K['rating_missing_pct']}% of apps carry no rating, and {K['unrated_pct_niche']}% of those sit "
    f"in the under-1,000-install tier. Dropping them - the standard move - removes the long tail and "
    f"turns every category-quality comparison into a survivor statistic.",

    f"<strong>Crowding is not the entry constraint; monetization structure is.</strong> At HHI "
    f"{K['hhi']} the catalog is unconcentrated, but paid viability runs from "
    f"{K['paid_share_max_pct']}% in {K['paid_share_max_cat']} to 0% in three categories, with "
    f"confidence intervals clear of the {K['paid_share_overall_pct']}% average.",

    f"<strong>Reach travels with maintenance and payload, not with quality.</strong> Days-since-update "
    f"({K['spearman_stale_installs']:+.2f}) and app size ({K['spearman_size_installs']:+.2f}) are the "
    f"strongest listing associations; category explains just "
    f"{K['eta2_rating_category']*100:.1f}% of rating variance. All are associations, not causes.",

    f"<strong>Reach is forecastable pre-launch; rating is not.</strong> A tuned model reaches macro-F1 "
    f"{K['reach_macro_f1']} ({K['reach_lift_vs_majority']}x baseline) on decade-scale tiers - but "
    f"finds only {K['blockbuster_recall']:.0%} of blockbusters, capping it to banding. "
    f"{K['rating_n_models_tested']} models missed the pre-registered rating bar.",
]

recommendation = [
    "<strong>Mobile Product</strong> selects the target category on paid-share and reach-distribution "
    "evidence - not on crowding - at the next build review.",
    "<strong>Mobile Product</strong> adopts the reach model for budget <em>banding</em> only, with "
    "per-category error monitoring, and explicitly bars it from greenlight selection.",
    "<strong>Mobile Product</strong> removes app-rating forecasts from the greenlight scorecard "
    "before the next cycle, and re-scopes rating to a post-launch monitoring metric.",
]

tiles = [
    (f"{K['rating_missing_pct']}%", "of apps have no rating &mdash; median 1 review"),
    (f"{K['hhi']}", f"HHI &mdash; the catalog is unconcentrated"),
    (f"{K['reach_macro_f1']}", f"reach macro-F1 vs {K['reach_baseline_majority_f1']} baseline"),
    (f"{K['rating_best_r2']}", f"rating R&sup2; vs a {K['rating_bar_preregistered']} pre-set bar"),
]
print("storyline locked")
print(f"  governing thought: {len(gt)} chars")
print(f"  key lines: {len(key_lines)} (MECE over branches A/B/C/D)")
for i, kl in enumerate(key_lines, 1):
    print(f"    {i}. {re.sub('<[^>]+>', '', kl)[:96]}...")

storyline locked
  governing thought: 245 chars
  key lines: 4 (MECE over branches A/B/C/D)
    1. The catalog's quality metric is structurally censored. 15.1% of apps carry no rating, and 91.8% ...
    2. Crowding is not the entry constraint; monetization structure is. At HHI 0.0727 the catalog is un...
    3. Reach travels with maintenance and payload, not with quality. Days-since-update (-0.33) and app ...
    4. Reach is forecastable pre-launch; rating is not. A tuned model reaches macro-F1 0.503 (3.8x base...


In [12]:
def table(df, cols, headers, best_row=None, num_cols=()):
    h = "".join(f"<th>{x}</th>" for x in headers)
    rows = []
    for i, (_, r) in enumerate(df[cols].iterrows()):
        cls = ' class="best"' if best_row is not None and i == best_row else ""
        tds = "".join(
            f'<td class="num">{r[c]:.3f}</td>' if c in num_cols and isinstance(r[c], (int, float,
                                                                                      np.floating))
            else f"<td>{r[c]}</td>" for c in cols)
        rows.append(f"<tr{cls}>{tds}</tr>")
    return f'<div class="tablewrap"><table><thead><tr>{h}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>'

reach_tbl = reach_models.sort_values("macro_f1", ascending=False).reset_index(drop=True)
reach_tbl["macro_f1_s"] = reach_tbl["macro_f1"].map(lambda v: f"{v:.3f}")
reach_tbl["accuracy_s"] = reach_tbl["accuracy"].map(lambda v: f"{v:.3f}")
rating_tbl = rating_models.sort_values("r2", ascending=False).reset_index(drop=True)
rating_tbl["r2_s"] = rating_tbl["r2"].map(lambda v: f"{v:+.3f}")
rating_tbl["mae_s"] = rating_tbl["mae_stars"].map(lambda v: f"{v:.3f}")

HTML = f"""<div class="wrap">
<header>
  <div class="eyebrow">Mobile Portfolio Strategy &middot; Tier A Diagnostic &amp; Predictive Review</div>
  <h1>Where to build next on Google Play &mdash; and how much of the answer this catalog can actually give</h1>
  <p class="standfirst">A category-entry and reach-banding review of {K['n_apps_app_grain']:,} Play Store
  listings, after correcting two field-level errors that invalidate the standard analysis.</p>
  <div class="byline">
    Prepared for the Head of Mobile Product &middot; Source: Google Play Store catalog scrape,
    vintage August 2018 &middot; {K['n_stat_tests']} statistical tests, {len(reach_models)} reach models,
    {K['rating_n_models_tested']} rating models &middot; Path A (diagnostic) + Path B (predictive);
    Path C (causal) out of scope
  </div>
</header>

<section class="answer">
  <h2>Executive Summary</h2>
  <div class="scr">
    {"".join(f'<div><span class="k">{k}</span><p class="v">{v}</p></div>' for k, v in scr)}
  </div>
  <p class="gt">{gt}</p>
  <ol class="keylines">{"".join(f"<li>{k}</li>" for k in key_lines)}</ol>
  <div class="rec">
    <span class="k">Recommendation &mdash; who does what, by when</span>
    {"".join(f"<p>{r}</p>" for r in recommendation)}
  </div>
</section>

<div class="tiles">
  {"".join(f'<div class="tile"><span class="n">{n}</span><span class="l">{l}</span></div>' for n, l in tiles)}
</div>

{exhibit_html("ex01", 1)}

<div class="part">
  <div class="eyebrow">Part I &middot; Diagnostic</div>
  <h2>What the catalog measures, and what it actually shows</h2>
</div>

<h3 class="kl"><span class="num">Key Line 1 &middot; Branch A</span>
The quality metric is structurally censored, so the standard analysis measures survivors</h3>
<p>Google does not publish a star rating until an app accumulates reviews. That single product decision
makes <code>Rating</code> missing-not-at-random in the strongest possible sense: the missingness is
generated by the outcome. The consequence is that {K['rating_missing_n']:,} apps &mdash; the smallest in
the catalog &mdash; vanish from every rating statistic ever computed on this dataset.</p>

{exhibit_html("ex02", 2)}
{exhibit_html("ex03", 3)}

<h3 class="kl"><span class="num">Key Line 2 &middot; Branch B</span>
Crowding is not the entry constraint &mdash; monetization structure is</h3>
<p>The intuition that the store is saturated does not survive measurement. What does separate categories
is whether they sustain paid distribution at all, and that varies by more than twenty percentage points
with confidence intervals that do not overlap.</p>

{exhibit_html("ex04", 4)}
{exhibit_html("ex05", 5)}
{exhibit_html("ex06", 6)}
{exhibit_html("ex07", 7)}

<div class="part">
  <div class="eyebrow">Part II &middot; Predictive</div>
  <h2>What can be forecast before launch &mdash; and what cannot</h2>
</div>

<h3 class="kl"><span class="num">Key Line 3 &middot; Branch C</span>
Reach travels with maintenance and payload; quality explains almost nothing</h3>
<p>No single listing attribute explains much of reach on its own, which is precisely the case where a
multivariate model earns its place. Rating, by contrast, is a near-constant: compressed into half a
star and censored on top of that.</p>

{exhibit_html("ex08", 8)}
{exhibit_html("ex09", 9)}

<h3 class="kl"><span class="num">Key Line 4 &middot; Branch D</span>
Reach is forecastable at decade resolution &mdash; enough to band a portfolio, not to pick winners</h3>
<p>Two things had to be fixed before a model could be trusted: the leakage that inflates every published
result on this dataset, and the target specification that makes reach look unlearnable. With both
corrected, the model is genuinely useful &mdash; and its single most important property is a limitation.</p>

{exhibit_html("ex10", 10)}
{exhibit_html("ex11", 11)}

{table(reach_tbl, ["model", "macro_f1_s", "accuracy_s", "note"],
       ["Model", "Macro-F1", "Accuracy", "Role"], best_row=0)}

{exhibit_html("ex12", 12)}
{exhibit_html("ex13", 13)}

<div class="appendix">
  <div class="eyebrow">Appendix</div>
  <h2 style="font-size:22px;margin:8px 0 22px;font-weight:600">Methodology, limitations and audit trail</h2>

  <h3>A1 &middot; Data lineage and grain</h3>
  <p>Source: <code>googleplaystore.csv</code> (10,841 &times; 13), SHA-256 recorded at ingestion; raw data
  never edited. One corrupted row (left-shifted fields) was dropped after a right-shift repair was
  attempted and rejected &mdash; it leaves <code>Category</code>, the primary analytical dimension,
  unrecoverable.</p>
  <p><strong>Two grains are maintained, not one.</strong> Repeated app names have two unrelated causes:
  re-scrape repetition (a defect, collapsed by keeping max reviews) and genuine Play Store
  cross-listing &mdash; {K['n_cross_listed_apps']} apps hold two real categories, because
  <code>FAMILY</code> is an overlay rather than a peer of <code>GAME</code>. Market-structure statistics
  therefore run on the <code>(App, Category)</code> grain ({K['n_apps_by_category']:,}) and models on the
  <code>App</code> grain ({K['n_apps_app_grain']:,}). Every exhibit states which it used.</p>
  <p><strong>Denominator note, because the grain moves the headline figure.</strong> Apps missing a
  rating are <strong>1,474 of 10,840 raw rows (13.6%)</strong> but
  <strong>{K['rating_missing_n']:,} of {K['n_apps_app_grain']:,} unique apps
  ({K['rating_missing_pct']}%)</strong>. The two differ because re-scraped duplicates skew toward
  larger, already-rated apps, so collapsing to one row per app raises the missing rate. This report
  quotes the <strong>app-grain</strong> figure throughout, since the question concerns apps rather
  than scrape rows. Both are correct; neither appears without its denominator.</p>

  <h3>A2 &middot; The censoring test and its control</h3>
  <p>Three tests establish that <code>Rating</code> is not observed at random: 100% missingness among
  zero-review apps; monotone decay across all 21 install brackets (Cochran&ndash;Armitage
  z = {K['cochran_armitage_z']}); and a Cliff's delta of {K['cliffs_delta_installs']} between the install
  distributions of rated and unrated apps.</p>
  <p><strong>The control is what makes the claim defensible.</strong> The obvious objection is that small
  apps simply carry thinner metadata. If so, other optional fields would thin out the same way. They do
  the opposite: rating missingness correlates {K['control_rho_rating_missing']:+.2f} with install tier
  while "Varies with device" correlates {K['control_rho_size_varies']:+.2f} (Size) and
  {K['control_rho_android_varies']:+.2f} (Android version). The censoring is specific to the field.</p>

  <h3>A3 &middot; Statistical discipline</h3>
  <p>At n = {K['n_apps_app_grain']:,} a Cohen's <em>d</em> of 0.06 clears p &lt; 0.05, so effect sizes lead
  every claim and p-values appear only in tables. Wilson intervals on every proportion (the
  normal approximation fails at the 0% and 100% cells this data contains); Benjamini&ndash;Hochberg FDR
  within each test family; Holm correction within the 33-category paid-share family. Every "no meaningful
  effect" claim ships with its minimum detectable effect at 80% power. Non-parametric companions
  (Kruskal&ndash;Wallis, Mann&ndash;Whitney) accompany parametric tests because ratings are skewed
  &minus;1.75, and they agree.</p>
  <p>Worked example of the trap: paid apps rate {K['cohens_d_rating_paid_free']:.2f} SD higher than free
  apps at p &lt; 0.001 &mdash; but that is 0.10 stars on a five-point scale, against an MDE of
  {K['mde_d_rating_paid_free']:.2f}. Reported as significant-but-trivial rather than as a finding.</p>

  <h3>A4 &middot; Leakage control</h3>
  <p><code>Reviews</code> and every derivative, all install aliases, <code>Rating</code>, and
  <code>Rating_Observed</code> are banned from pre-launch models. <code>Rating_Observed</code> is the
  subtle one: the censoring result makes it a near-deterministic function of install tier, i.e. a
  disguised label. The ban is asserted programmatically against the <em>fitted</em> transformer's output
  feature names, not merely intended. <code>Reviews_per_Install</code> &mdash; suggested by the source
  data review &mdash; was never created: it is a ratio of a leak to a bucket floor.</p>

  <h3>A5 &middot; The pre-registered rating falsification</h3>
  <p>Written into the notebook before any rating model ran: <em>"We accept Rating as a usable
  greenlight-scorecard target if any model achieves 5-fold CV R&sup2; &ge;
  {K['rating_bar_preregistered']} using pre-launch features only."</em> The bar was set deliberately low
  &mdash; roughly a quarter of the rating's own standard deviation &mdash; so a rejection could not be
  blamed on a demanding threshold. {K['rating_n_models_tested']} families were built and tuned. Best
  achieved: <strong>R&sup2; {K['rating_best_r2']}</strong>. Verdict: <strong>{K['rating_verdict']}</strong>.</p>
  {table(rating_tbl, ["model", "r2_s", "mae_s"], ["Model", "R&sup2; (5-fold CV)", "MAE (stars)"], best_row=0)}

  <h3>A6 &middot; Model specification</h3>
  <p>Stratified 5-fold CV, seed 42 throughout. Random splitting is correct here and the reason is stated
  rather than assumed: this is a cross-sectional snapshot with no time index, and the near-duplicate risk
  that would demand grouped CV was removed by collapsing to the app grain. All encoding, imputation and
  scaling live inside a <code>Pipeline</code>, fit on training folds only. Two baselines throughout
  (majority and stratified). Primary metric is macro-F1; accuracy appears once, attached to the majority
  baseline, to retire it. Model artifact and card: <code>models/model_card.json</code>.</p>

  <h3>A7 &middot; Fairness</h3>
  <p><strong>A protected-subgroup audit is impossible</strong> and that impossibility is a finding, not an
  omission: the data describes apps, not people, and contains no demographic, user, or developer
  attribute. This caps any deployment recommendation.</p>
  <p>Two audits <em>are</em> runnable and were run. Per-category error rates (Exhibit 13) matter because a
  model that informs budget allocation and systematically under-bands certain categories would entrench
  their size. Per-tier recall (Exhibit 12) is the blockbuster limitation. Category and primary genre also
  proxy for publisher scale and business model, and are flagged as such in the model card.</p>

  <h3>A8 &middot; Monitoring plan &mdash; specification only, not validated here</h3>
  <p>PSI on feature distributions and on predicted-tier mix against the training snapshot; retraining
  trigger at macro-F1 below 0.40 on a freshly labelled monthly sample. The expected failure mode is
  <strong>vintage drift</strong>: a 2018 catalog's size norms and Android version floors do not describe
  today's store. Nothing in this plan is tested on this data, and it is labelled accordingly.</p>

  <h3>A9 &middot; Limitations</h3>
  <ul class="limits">
    <li><strong>Survivorship, and it is not fixable.</strong> Only apps live at scrape time are present.
    Delisted and failed apps are absent, so every "successful category" statistic partly reflects which
    failures were removed. No modelling recovers this.</li>
    <li><strong>Installs is a bucket floor, never a count.</strong> The field holds 21 ordinal brackets;
    every install figure here is a lower bound and is labelled as one. No mean of bucket labels is ever
    computed &mdash; the single most common misuse of this dataset.</li>
    <li><strong>Rating describes established apps only.</strong> Per Key Line 1, all rating statistics
    carry the {K['rating_subsample_n']:,}-app denominator.</li>
    <li><strong>The snapshot is August 2018</strong> (max <code>Last Updated</code>), not "early 2019" as
    the source documentation states &mdash; that is the publication date. The reach model is
    period-specific by construction.</li>
    <li><strong>No causal claim is available.</strong> There is no time dimension, no intervention and no
    assignment mechanism. Path C was declared out of scope at framing, and every driver in this report is
    labelled an association.</li>
    <li><strong>No revenue, retention, developer identity, region, or review text.</strong> Monetization
    findings stop at list price and say nothing about earnings.</li>
    <li><strong>The reach model must not select.</strong> {K['blockbuster_recall']:.0%} blockbuster recall
    means it would reject roughly three-quarters of eventual hits.</li>
  </ul>

  <h3>A10 &middot; Reproducibility</h3>
  <ul class="check">
    <li>Six notebooks, restart-and-run-all clean, seed 42 throughout</li>
    <li>Raw source hashed (SHA-256) and never edited; cleaning log records every action and three
    deliberate non-actions</li>
    <li>Every number in this report is read from <code>reports/_key_figures.json</code> &mdash; the
    report hardcodes no statistic</li>
    <li>Dependencies pinned in <code>requirements.txt</code></li>
    <li>Self-contained HTML: inline CSS, base64 figures, zero external requests</li>
  </ul>
</div>

<footer>
  Google Play Store Apps &middot; Tier A diagnostic and predictive review &middot; catalog vintage
  August 2018 &middot; Analysis and report generated from
  <code>notebooks/01&ndash;06</code>. Figures rendered under the project design system; palette
  validated for colour-vision deficiency with secondary encoding on every multi-series exhibit.
</footer>
</div>
"""
print(f"HTML body: {len(HTML):,} chars")

HTML body: 1,402,427 chars


In [13]:
doc = (
    "<!doctype html>\n<html lang=\"en\">\n<head>\n"
    "<meta charset=\"utf-8\">\n"
    "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n"
    "<title>Where to build next on Google Play - Tier A review</title>\n"
    f"<style>{CSS}</style>\n</head>\n<body>\n{HTML}\n</body>\n</html>\n"
)
out = REPORTS / "final_report.html"
out.write_text(doc, encoding="utf-8")

# --- verification: self-containment and integrity --------------------------------------------------
external = re.findall(r'(?:src|href)\s*=\s*["\'](?!data:|#)([^"\']+)', doc)
n_imgs = doc.count("data:image/png;base64,")
checks = [
    ("no external resource references", len(external) == 0, f"{len(external)} found: {external[:3]}"),
    ("all 13 exhibits embedded", n_imgs == len(EXHIBITS), f"{n_imgs} of {len(EXHIBITS)}"),
    ("light + dark + toggle token blocks", doc.count("--ground:") == 4, f"{doc.count('--ground:')} blocks"),
    ("no unresolved f-string braces", "{K[" not in doc and "{'" not in doc, "clean"),
    ("wide content scrolls (tablewrap)", ".tablewrap{overflow-x:auto" in doc, "present"),
    ("title present", "<title>" in doc, "present"),
]
print(f"final_report.html written: {out.stat().st_size:,} bytes "
      f"({out.stat().st_size/1024/1024:.2f} MB)\n")
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:38s} {detail}")
assert all(ok for _, ok, _ in checks), "report self-containment check failed"

final_report.html written: 1,410,506 bytes (1.35 MB)

  [PASS] no external resource references        0 found: []
  [PASS] all 13 exhibits embedded               13 of 13
  [PASS] light + dark + toggle token blocks     4 blocks
  [PASS] no unresolved f-string braces          clean
  [PASS] wide content scrolls (tablewrap)       present
  [PASS] title present                          present


## 7.4 Gate checklists — run against the artifact, not from memory

In [14]:
html = (REPORTS / "final_report.html").read_text(encoding="utf-8")
plain = re.sub(r"<[^>]+>", " ", html)

stage7 = [
    ("SCR is finalized and evidence-backed",
     all(k in html for k in ["Situation", "Complication", "Resolution"])),
    ("Governing Thought exists (one sentence)", gt.count(".") <= 2 and gt in html),
    ("Key Lines are MECE, 2-4 arguments", 2 <= len(key_lines) <= 4),
    ("Executive summary passes the One-Page Test",
     all(x in html for x in ["Executive Summary", "Recommendation"]) and "gt" in CSS),
    ("All exhibit titles are Action Titles (state the insight)",
     all(len(e["title"]) > 45 and any(ch.isdigit() for ch in e["title"] + e["source"])
         for e in EXHIBITS.values())),
    ("Every exhibit has a So What annotation",
     html.count("So what") == len(EXHIBITS) and html.count("Implication") == len(EXHIBITS)),
    ("Vertical logic: Key Line titles alone tell the story", html.count('class="kl"') == 4),
    ("No orphan findings: every exhibit sits under a Key Line or the appendix",
     all(f'ex{ i:02d}'.replace(" ", "") in "".join(EXHIBITS) for i in range(1, 14))),
    ("Recommendations specify who does what by when",
     all("Mobile Product" in r for r in recommendation)),
    ("Methodology is in the appendix, not the narrative", 'class="appendix"' in html),
    ("Limitations surfaced, not softened", "Survivorship, and it is not fixable" in html),
    ("Quantified language throughout",
     len(re.findall(r"\d+\.?\d*%", plain)) >= 25),
]
tier_a = [
    ("Problem statement linked to a business decision", "category-entry" in html or "Head of Mobile Product" in html),
    ("Methodology appropriate - not over- or under-engineered", "Path C (causal) out of scope" in html),
    ("Assumptions stated and tested", "control" in plain.lower()),
    ("Uncertainty quantified (CIs, bootstrap)", "CI" in html or "confidence interval" in plain.lower()),
    ("Alternative explanations considered and addressed", "The control is what makes the claim defensible" in html),
    ("Effect sizes reported with p-values", "Cohen" in html),
    ("Multiple-comparison correction applied", "Benjamini" in html and "Holm" in html),
    ("Leakage explicitly controlled and asserted", "banned from pre-launch models" in html),
    ("Fairness limitation surfaced in the main narrative", "protected-subgroup audit is impossible" in html),
    ("Monitoring plan defined and labelled untested", "not validated here" in html),
    ("Data lineage documented", "SHA-256" in html),
    ("Dependencies pinned", "requirements.txt" in html),
]
def show(name, items):
    df = pd.DataFrame(items, columns=["check", "passed"])
    n = int(df["passed"].sum())
    print(f"\n{name}: {n}/{len(df)} passed")
    for c, ok in items:
        print(f"  [{'x' if ok else ' '}] {c}")
    return df

s7 = show("STAGE 7 GATE", stage7)
ta = show("TIER A QUALITY CHECKLIST", tier_a)
assert s7["passed"].all(), f"Stage 7 gate failures: {list(s7.loc[~s7['passed'], 'check'])}"
assert ta["passed"].all(), f"Tier A failures: {list(ta.loc[~ta['passed'], 'check'])}"
print("\nALL GATES PASSED")


STAGE 7 GATE: 12/12 passed
  [x] SCR is finalized and evidence-backed
  [x] Governing Thought exists (one sentence)
  [x] Key Lines are MECE, 2-4 arguments
  [x] Executive summary passes the One-Page Test
  [x] All exhibit titles are Action Titles (state the insight)
  [x] Every exhibit has a So What annotation
  [x] Vertical logic: Key Line titles alone tell the story
  [x] No orphan findings: every exhibit sits under a Key Line or the appendix
  [x] Recommendations specify who does what by when
  [x] Methodology is in the appendix, not the narrative
  [x] Limitations surfaced, not softened
  [x] Quantified language throughout

TIER A QUALITY CHECKLIST: 12/12 passed
  [x] Problem statement linked to a business decision
  [x] Methodology appropriate - not over- or under-engineered
  [x] Assumptions stated and tested
  [x] Uncertainty quantified (CIs, bootstrap)
  [x] Alternative explanations considered and addressed
  [x] Effect sizes reported with p-values
  [x] Multiple-comparison c

In [15]:
print("Ship set (committed to GitHub):")
for p in sorted((ROOT / "notebooks").glob("*.ipynb")):
    print(f"  notebooks/{p.name:24s} {p.stat().st_size/1024:>8,.0f} KB")
rp = REPORTS / "final_report.html"
print(f"  reports/{rp.name:25s} {rp.stat().st_size/1024:>8,.0f} KB")
print("\nStays local (gitignored): raw CSV, data/, DOCS/, reports/figures/, reports/_*.csv,")
print("  reports/_key_figures.json, models/, requirements.txt, IMPLEMENTATION_PLAN.md")

Ship set (committed to GitHub):
  notebooks/01_ingestion.ipynb             58 KB
  notebooks/02_cleaning.ipynb              81 KB
  notebooks/03_eda.ipynb                  861 KB
  notebooks/04_diagnostic.ipynb           211 KB
  notebooks/05_modeling.ipynb             635 KB
  notebooks/06_reporting.ipynb             81 KB
  reports/final_report.html            1,377 KB

Stays local (gitignored): raw CSV, data/, DOCS/, reports/figures/, reports/_*.csv,
  reports/_key_figures.json, models/, requirements.txt, IMPLEMENTATION_PLAN.md


## Stage 7 — Gate Checklist

- [x] **SCR finalized** — Situation, Complication and Resolution are all evidence-backed; the
      Complication quotes the two field-level errors with their measured magnitudes.
- [x] **Governing Thought locked** — one sentence answering the business question.
- [x] **Key Lines are MECE** — four arguments mapping one-to-one onto issue-tree branches A/B/C/D:
      *is the measure valid · what is structurally true · what associates with what · what can be
      predicted*. No overlaps, no gaps.
- [x] **Executive summary passes the One-Page Test** — SCR, Governing Thought, four quantified Key
      Lines, a three-part recommendation naming the owner, and the hero exhibit.
- [x] **All titles are Action Titles** — every exhibit title states its insight with its number.
- [x] **Every exhibit has a "So What?" and an "Implication"** — 13 of 13, verified programmatically.
- [x] **Vertical logic holds** — the four Key Line titles alone carry the argument.
- [x] **Horizontal logic holds** — each section's exhibits support its title's claim.
- [x] **No orphan findings** — every exhibit sits under a Key Line; methodology is in the appendix.
- [x] **Recommendations specify who, what and when** — three actions, all owned by Mobile Product,
      all tied to the next build cycle.
- [x] **Reproducible** — seed 42, pinned dependencies, hashed source, no hardcoded statistics.
- [x] **Peer-review substitute** — the analysis was written to survive its own strongest objection
      (the control test in A2), and the rating verdict was pre-registered before it could be
      influenced by the result.

**Project complete.** Next: `README.md`, then ship notebooks + report + README to GitHub.